# Aula 01: Algoritmos Gulosos (Greedy Algorithms)

No Módulo 01, focamos em métodos exatos que garantem a melhor solução possível (ótimo global). No entanto, para problemas de grande escala, o tempo de processamento desses métodos pode ser proibitivo.

Entramos agora no mundo dos **Métodos Aproximados**. O algoritmo mais intuitivo desse grupo é o **Algoritmo Guloso**.

## 1. O que é um Algoritmo Guloso?

Um algoritmo guloso segue o princípio de fazer a **escolha localmente ótima** em cada etapa, com a esperança de encontrar o ótimo global. 

### Características:
1. **Míope**: Ele não olha para o futuro; apenas para o benefício imediato.
2. **Irreversível**: Uma vez feita a escolha, ela não é alterada (não há *backtracking*).
3. **Rápido**: Geralmente possui complexidade de tempo muito baixa.

**Cuidado**: Nem sempre a escolha gulosa leva à melhor solução global. Veremos isso na prática.

## 2. TSP: Heurística do Vizinho Mais Próximo (Nearest Neighbor)

Uma das formas mais clássicas de resolver o TSP sem usar Programação Linear é:
1. Comece em uma cidade qualquer.
2. Vá para a cidade não visitada mais próxima.
3. Repita até visitar todas as cidades.
4. Retorne à cidade de origem.

In [ ]:
import math
import matplotlib.pyplot as plt
import json

def dist(c1, c2):
    return math.sqrt((c1['x'] - c2['x'])**2 + (c1['y'] - c2['y'])**2)

# Carregar dados
try:
    with open('data/greedy_tsp.json', 'r') as f:
        data = json.load(f)['instancia_greedy']
        cidades = data['cidades']
except:
    cidades = [
        {"id": 0, "x": 0, "y": 0}, {"id": 1, "x": 10, "y": 0},
        {"id": 2, "x": 10, "y": 10}, {"id": 3, "x": 0, "y": 10},
        {"id": 4, "x": 5, "y": 5}, {"id": 5, "x": 2, "y": 8},
        {"id": 6, "x": 8, "y": 2}
    ]

def solve_nn(lista_cidades, start_node=0):
    unvisited = lista_cidades.copy()
    current = unvisited.pop(start_node)
    tour = [current]
    total_dist = 0
    
    while unvisited:
        next_node = min(unvisited, key=lambda c: dist(current, c))
        total_dist += dist(current, next_node)
        tour.append(next_node)
        unvisited.remove(next_node)
        current = next_node
    
    total_dist += dist(tour[-1], tour[0]) # Retorno
    return tour, total_dist

tour_greedy, d_greedy = solve_nn(cidades)
print(f"Distância Total (Guloso): {d_greedy:.2f}")

# Visualização
plt.figure(figsize=(8, 6))
xs = [c['x'] for c in tour_greedy] + [tour_greedy[0]['x']]
ys = [c['y'] for c in tour_greedy] + [tour_greedy[0]['y']]
plt.plot(xs, ys, 'o-', color='green', label='Rota Gulosa')
for c in cidades:
    plt.annotate(c['id'], (c['x'], c['y']), xytext=(5, 5), textcoords='offset points')
plt.title(f"TSP Guloso: Vizinho Mais Próximo (Dist: {d_greedy:.2f})")
plt.legend()
plt.grid(True)
plt.show()

## 3. O Problema da Mochila (Knapsack) Guloso

Para a mochila, uma estratégia gulosa comum é escolher os itens com a melhor **densidade de valor** (Valor / Peso).

### Exemplo:
Mochila com capacidade 20kg.
- Item A: 10kg, R$ 60 (R$ 6/kg)
- Item B: 15kg, R$ 75 (R$ 5/kg)
- Item C: 10kg, R$ 50 (R$ 5/kg)

**Escolha Gulosa**: Item A (melhor custo-benefício). Sobram 10kg. Podemos pegar o Item C. 
**Resultado**: R$ 110. (Neste caso, é o ótimo, mas nem sempre!)

In [ ]:
itens = [
    {"id": "A", "valor": 60, "peso": 10},
    {"id": "B", "valor": 75, "peso": 15},
    {"id": "C", "valor": 50, "peso": 10}
]
capacidade = 20

# Ordenar por valor/peso
itens_ordenados = sorted(itens, key=lambda x: x['valor']/x['peso'], reverse=True)

mochila = []
peso_atual = 0
valor_total = 0

for item in itens_ordenados:
    if peso_atual + item['peso'] <= capacidade:
        mochila.append(item['id'])
        peso_atual += item['peso']
        valor_total += item['valor']

print(f"Itens na mochila: {mochila}")
print(f"Valor Total: R$ {valor_total}")
print(f"Peso Final: {peso_atual}kg")

## 4. Laboratório: Quando o Guloso Falha?

O problema dos algoritmos gulosos é que eles podem ficar presos em "ótimos locais".

### Exercício 1: Mudança de Ponto de Partida
No código do TSP (Seção 2), mude o `start_node` de **0** para **4**. 
- A distância total mudou?
- O algoritmo guloso garante a mesma solução independente de onde começa?

### Exercício 2: O contra-exemplo da Mochila
Imagine uma mochila de 10kg e os itens:
- Item 1: 6kg, R$ 10 (R$ 1.66/kg)
- Item 2: 5kg, R$ 7 (R$ 1.4/kg)
- Item 3: 5kg, R$ 7 (R$ 1.4/kg)

**Pergunta**: O que o algoritmo guloso escolheria? Qual seria a escolha manual que daria mais lucro? Por que o guloso falhou aqui?